## Import Libraries

In [26]:
# Import necessary libraries
import pandas as pd
import ast
import torch
from transformers import AutoTokenizer, AutoModel
from torch.nn.functional import cosine_similarity

## Load Datasets

In [27]:
# Load movie metadata and keywords
movies = pd.read_csv("movies_metadata.csv", low_memory=False)
keywords = pd.read_csv("keywords.csv")

# Select relevant columns and rename id -> movieid
movies = movies[['id', 'genres', 'overview', 'title']].rename(columns={'id': 'movieid'})
keywords = keywords[['id', 'keywords']].rename(columns={'id': 'movieid'})

## Parse JSON Columns

In [28]:
# Function to parse genres JSON and join names
def parse_genres(text):
    try:
        genres = ast.literal_eval(text)
        return ", ".join([g['name'] for g in genres])
    except:
        return ""

# Apply parsing to movies dataframe
movies['genres'] = movies['genres'].apply(parse_genres)

# Function to parse keywords JSON and join names
def parse_keywords(text):
    try:
        kws = ast.literal_eval(text)
        return ", ".join([k['name'] for k in kws])
    except:
        return ""

# Apply parsing to keywords dataframe
keywords['keywords'] = keywords['keywords'].apply(parse_keywords)

## Merge Movies & Keywords

In [29]:
movies['movieid'] = pd.to_numeric(movies['movieid'], errors='coerce').astype('Int64')
keywords['movieid'] = pd.to_numeric(keywords['movieid'], errors='coerce').astype('Int64')

movies = movies.dropna(subset=['movieid'])
keywords = keywords.dropna(subset=['movieid'])
movies.info()

keywords.info()

<class 'pandas.core.frame.DataFrame'>
Index: 45463 entries, 0 to 45465
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   movieid   45463 non-null  Int64 
 1   genres    45463 non-null  object
 2   overview  44509 non-null  object
 3   title     45460 non-null  object
dtypes: Int64(1), object(3)
memory usage: 1.8+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46419 entries, 0 to 46418
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   movieid   46419 non-null  Int64 
 1   keywords  46419 non-null  object
dtypes: Int64(1), object(1)
memory usage: 770.8+ KB


In [31]:
import numpy as np
# Merge on movieid to get overview, genres, keywords together
df = movies.merge(keywords, on='movieid', how='inner')
df[['overview', 'genres', 'keywords', 'title']] = df[['overview', 'genres', 'keywords', 'title']].replace('', np.nan)
# Drop rows with missing overview
df = df.dropna(subset=['overview', 'genres', 'keywords', 'title'], how='any')

## Create Structured "Soup"

In [32]:
# Combine overview, genres, keywords into a structured string
def create_soup(row):
    return (
        f"Overview: {row['overview']}. "
        f"Genres: {row['genres']}. "
        f"Keywords: {row['keywords']}."
    )

# Apply to dataframe
df['soup'] = df.apply(create_soup, axis=1)

## Load Pretrained Sentence Transformer

In [33]:
# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
model = AutoModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Embed Function

In [34]:
# Convert text into embeddings
def embed(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
    # Mean pooling + normalization
    embedding = outputs.last_hidden_state.mean(dim=1)
    embedding = torch.nn.functional.normalize(embedding, dim=1)
    return embedding

## Build Embedding Matrix

In [35]:
# Compute embeddings for all movies
embeddings = torch.vstack([embed(text) for text in df['soup']])

# Check shape
print(embeddings.shape)  # (num_movies, 384)

torch.Size([31088, 384])


In [36]:
# ## Recommend Function
def recommend(movie_idx, top_k=2):
    """
    Recommend top_k similar movies for a given movie index.
    """
    target = embeddings[movie_idx]
    sims = cosine_similarity(target.unsqueeze(0), embeddings)
    top_indices = torch.topk(sims, top_k + 1).indices[1:]
    return df.iloc[top_indices]

# ## Genre Overlap Metric
def genre_overlap(movie_idx, rec_indices, log_prefix="genre_overlap"):
    target_genres = set(df.iloc[movie_idx]['genres'].split(", "))
    scores = []
    for idx in rec_indices:
        rec_genres = set(df.iloc[idx]['genres'].split(", "))
        overlap = len(target_genres & rec_genres) / len(target_genres) if target_genres else 0
        scores.append(overlap)
        print(f"Target Genres: {target_genres}, Recommended Genres: {rec_genres}, Overlap: {overlap:.2f}")
    
    avg_overlap = sum(scores) / len(scores) if scores else 0
    # Log with unique name to prevent overwrite
    print(f"{log_prefix}_{random.randint(0,9999)}", avg_overlap)
    return avg_overlap



# ## Intra-Similarity Metric
def intra_similarity(indices, log_prefix="intra_sim"):
    sims = []
    for i in indices:
        for j in indices:
            if i < j:  # avoid repeating symmetric comparisons
                sims.append(cosine_similarity(
                    embeddings[i].unsqueeze(0),
                    embeddings[j].unsqueeze(0)
                ).item())
    avg_sim = sum(sims) / len(sims) if sims else 0
    # Log with unique name
    print(f"{log_prefix}_{random.randint(0,9999)}", avg_sim)
    return avg_sim

In [37]:
import random
# Pick 3 random movie indices
movie_indices = random.sample(range(len(df)), 3)

for i, movie_idx in enumerate(movie_indices):
    print(f"\n=== Movie {i+1} (Index: {movie_idx}) ===")
    print("Movie Title:", df.iloc[movie_idx]['title'])
    
    # Get recommendations
    recommendations = recommend(movie_idx, top_k=5)
    print("Recommended Movies:")
    print(recommendations[['title', 'genres', 'keywords', 'overview']])
    
    rec_indices = recommendations.index.tolist()
    
    # Genre Overlap
    print("Genre Overlap:", genre_overlap(movie_idx, rec_indices, log_prefix=f"genre_overlap_{movie_idx}"))
    
    # Embedding Cohesion
    print("Embedding Cohesion:", intra_similarity(rec_indices, log_prefix=f"intra_sim_{movie_idx}"))


=== Movie 1 (Index: 9438) ===
Movie Title: Mad Dog Morgan
Recommended Movies:
                                  title                          genres  \
7257                          Ned Kelly          Drama, Action, Western   
13343                  Gun The Man Down                 Western, Action   
9981     The Chant of Jimmie Blacksmith           Crime, Drama, History   
4667   The Return of a Man Called Horse          Action, Drama, Western   
21497          Ain't Them Bodies Saints  Crime, Drama, Romance, Western   

                                                keywords  \
7257   bank robber, australian, outlaw, based on true...   
13343                                        bank robber   
9981   based on novel, biography, australia, based on...   
4667   based on novel, conflict, wild west, sioux tri...   
21497  1970s, river, texas, shooting, outlaw, escape ...   

                                                overview  
7257   Unable to support his family in the Austral